# 1. Configuração do ambiente e leitura da base

### 1.1 Instalação das bibliotecas

As dependências do projeto estão em `requirements.txt` (versões exatas em `requirements-lock.txt`). O passo a passo do ambiente está no `README.md`.

As funções reutilizáveis deste notebook ficam no pacote `equifax` (pasta `src/equifax/`), instalado com `pip install -e .`:

| Módulo | O que tem |
|---|---|
| `config.py` | caminhos das pastas e arquivos e a semente `RANDOM_SEED` |
| `selecao.py` | lista dos códigos especiais das VPs (`CODIGOS_ESPECIAIS`) |
| `tratamento.py` | conferência do estado, formatação do CEP e categoria "Sem informação" |
| `validacao.py` | safras de desenvolvimento (`SAFRAS_DEV`) |
| `viz.py` | tabelas e gráficos da EDA e `salvar_figura` |

In [ ]:
# Instalação das dependências (rodar uma vez, com o .venv ativo)
%pip install -q -r ../requirements-lock.txt
%pip install -q -e ..

### 1.2 Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from equifax.config import DATA_PROCESSED, DES_PATH, RANDOM_SEED, TABLES
from equifax.selecao import CODIGOS_ESPECIAIS
from equifax.tratamento import ORDEM_RENDAS, checar_estados, formatar_cep, preencher_sem_informacao
from equifax.validacao import SAFRAS_DEV
from equifax.viz import (cruzar_com_adimplencia, cruzar_com_renda_e_adimplencia,
                         distribuicao_frequencia, salvar_figura)

### 1.3 Semente aleatória

Semente fixa em `42` para garantir reprodutibilidade nos modelos e amostragens. O valor é definido uma única vez, em `src/equifax/config.py`, e importado aqui. Portanto, sempre passaremos `RANDOM_SEED` explicitamente:

| Ferramenta                                                                                     | Parâmetro |
|------------------------------------------------------------------------------------------------|---|
| `df.sample(...)`, `train_test_split`, `StratifiedKFold(shuffle=True)`, modelos do scikit-learn | `random_state=RANDOM_SEED` |
| LightGBM / XGBoost                                                                             | `random_state=RANDOM_SEED` |
| CatBoost                                                                                       | `random_seed=RANDOM_SEED` |
| NumPy (geração nova)                                                                           | usar o `rng` abaixo |

In [ ]:
# RANDOM_SEED vem de equifax.config (o mesmo valor para todo o projeto)
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

### 1.4 Leitura das bases

O caminho da base vem de `config.DES_PATH` (`data/raw/base_des.csv`), então o notebook roda igual no computador de qualquer membro. A OOT não é carregada neste notebook.

In [ ]:
df_des = pd.read_csv(DES_PATH, sep=';')

print(f'df_des: {df_des.shape[0]} linhas x {df_des.shape[1]} colunas')

# 1 - adimplente (BP)
# 0 - inadimplente (MP)

### 1.5 Salvamento das figuras

Todo gráfico da EDA é salvo em `reports/figures/` por `salvar_figura(...)` para reaproveitarmos as imagens para o relatório. A função fica em `src/equifax/viz.py` e usa a pasta definida em `config.FIGURES`.

# 2. Análise exploratória

### 2.1 Panorama geral da base

In [ ]:
print(f"Info da base:")
df_des.info()
print()
display(df_des.dtypes.value_counts())
print()

vps = []
for col in df_des.columns:
    if 'VP' in col:
        vps.append(col)

# Save em CSV da tabela describe() das VPs para análise completa
df_des[vps].describe().T.to_csv(TABLES / 'describe_vps.csv')
df_des[vps].describe().T.head(20)

### 2.2 Formatação do CEP

A coluna `cep8` vem como tipo int no CSV, então os zeros à esquerda se perdem (ex.: `01310100` vira `1310100`). Por isso, convertemos ela para valores de 8 dígitos.

- cep8 == 0 vira nulo <NA>, ao invés de '00000000'.
- A OOT recebe a mesma função no pipeline do Passo 10, porque ela não é carregada neste notebook.
- A função `formatar_cep` fica em `src/equifax/tratamento.py`.

In [ ]:
df_des['cep8'] = formatar_cep(df_des['cep8'])

# Conferência sem mostrar CEPs: tamanho dos CEPs formatados (<NA> = CEP ausente)
df_des['cep8'].str.len().value_counts(dropna=False)

### 2.4 Valores nulos

In [ ]:
for col in df_des.columns:
    qtd_nan = df_des[col].isna().sum()
    if qtd_nan > 0:
        print(f'Nulos em {col}: {qtd_nan}')

Fora das VPs, os nulos de `faixa_idade` e `cep8` viram a categoria 'Sem informação', pois a falta do dado pode ter potencial discriminativo de bons e maus pagadores.

In [ ]:
cols_nulos = ['cep8', 'faixa_idade']

print('Nulos antes:', df_des[cols_nulos].isna().sum().to_dict())
print()
df_des = preencher_sem_informacao(df_des, cols_nulos)

print('faixa_idade = "Sem informação":', (df_des['faixa_idade'] == 'Sem informação').sum())
print('cep8 = "Sem informação":', (df_des['cep8'] == 'Sem informação').sum())

### 2.5 Códigos especiais nas VPs

O panorama inicial da base com a tabela `describe()` (analisado mais a fundo após o salvamento em CSV) apontou valores fixos que parecem códigos especiais: `-1`, `-2` e `9999`. Analisaremos:

- se os valores especiais são ou não valores reais em cada VP onde aparecem
- se a quantidade desses valores por linha está ligada à inadimplência

A lista `CODIGOS_ESPECIAIS` vem de `src/equifax/selecao.py`

In [ ]:
vps = [col for col in df_des.columns if col.startswith('VP')]
print(f'{len(vps)} VPs')

# % das linhas com cada código em cada VP
linhas = []
for col in vps:
    linha = {'VP': col}
    for codigo in CODIGOS_ESPECIAIS:
        linha[codigo] = (df_des[col] == codigo).mean()
    linhas.append(linha)
pct_codigos = pd.DataFrame(linhas).set_index('VP')

resumo_codigos = pd.DataFrame({
    'VPs com o código': (pct_codigos > 0).sum(),
    'VPs com mais de 50% das linhas': (pct_codigos > 0.5).sum(),
    'Maior %': pct_codigos.max(),
    'VP do maior %': pct_codigos.idxmax(),
})
resumo_codigos.index.name = 'Código'
display(resumo_codigos.style.format({'Maior %': '{:.1%}'}))

In [ ]:
# Só as VPs em que os códigos somam pelo menos 1% das linhas
com_codigo = pct_codigos[pct_codigos.sum(axis=1) >= 0.01]
com_codigo = com_codigo.loc[com_codigo.sum(axis=1).sort_values().index] * 100

ax = com_codigo.plot(kind='barh', stacked=True, figsize=(10, max(5, len(com_codigo) * 0.25)),
                     color=['steelblue', 'darkorange', 'firebrick'])
ax.axvline(50, color='gray', linestyle='--')
ax.set_xlim(0, 100)

plt.title('% das linhas com código especial em cada VP')
plt.xlabel('% das linhas')
plt.ylabel('VP')
plt.legend(title='Código')
plt.grid(axis='x', color="gray", alpha=.5)
plt.tight_layout()
salvar_figura('codigos_especiais_por_vp')
plt.show()

**- Valores especiais são ou não valores reais em cada VP onde aparecem**

Código: fica fora da faixa dos valores normais. Exemplo: `-1` em uma VP que começa em 0, ou um `9999` numa VP cujos valores normais são pequenos \
Valor real: se o valor cai dentro da faixa

In [ ]:
n_negativos = pd.Series({col: df_des.loc[df_des[col] < 0, col].nunique() for col in vps})
print(f'Maior número de valores negativos diferentes numa VP: {n_negativos.max()}')

linhas = []
for codigo in CODIGOS_ESPECIAIS:
    for col in pct_codigos.index[pct_codigos[codigo] > 0]:
        normais = df_des.loc[(df_des[col] >= 0) & ~df_des[col].isin(CODIGOS_ESPECIAIS), col]
        fora_da_faixa = codigo < normais.min() or codigo > normais.max()
        linhas.append({'Código': codigo, 'VP': col,
                       '% das linhas': pct_codigos.loc[col, codigo],
                       'Menor normal': normais.min(),
                       'Maior normal': normais.max(),
                       'Situação': 'código' if fora_da_faixa else 'valor real'})
situacao_codigos = pd.DataFrame(linhas)

display(pd.crosstab(situacao_codigos['Código'], situacao_codigos['Situação']))

# VPs em que o valor cai dentro da faixa dos valores normais
valores_reais = situacao_codigos[situacao_codigos['Situação'] == 'valor real']
display(valores_reais.set_index(['Código', 'VP']).style.format(
    {'% das linhas': '{:.4%}', 'Menor normal': '{:.0f}', 'Maior normal': '{:.0f}'}))

**Relação entre valores especiais por linha e inadimplência**

Para cada cliente, contamos em quantas VPs ele tem `-1` (entre as VPs em que o `-1` é código) e dividimos os clientes em quartis dessa contagem com `qcut`: quatro grupos com cerca de 25% dos clientes em cada. Cada barra mostra a inadimplência de um grupo.

O mesmo é feito com o `-2`. Para o `9999`, basta comparar quem tem e quem não tem o código na VP em que ele é código. Usamos só as safras de desenvolvimento (202301 a 202308), para deixar a OOT interna fora das análises que usam o target.

In [ ]:
dev = df_des['safra'].isin(SAFRAS_DEV)

for codigo in [-1, -2]:
    vps_do_codigo = situacao_codigos.loc[(situacao_codigos['Código'] == codigo)
                                         & (situacao_codigos['Situação'] == 'código'), 'VP']
    n = (df_des.loc[dev, vps_do_codigo] == codigo).sum(axis=1)

    # Quartis da contagem; o rótulo de cada grupo é o menor e o maior número de VPs dele
    quartil = pd.qcut(n, 4, labels=False, duplicates='drop')
    limites = n.groupby(quartil).agg(['min', 'max'])
    rotulos = [f'{a} a {b}' if a < b else f'{a}' for a, b in zip(limites['min'], limites['max'])]

    coluna = f'VPs com {codigo}'
    contagem = pd.DataFrame({coluna: quartil.map(dict(enumerate(rotulos))),
                             'adimplencia': df_des.loc[dev, 'adimplencia']})

    nome = str(codigo).replace('-', 'menos')
    cruzar_com_adimplencia(contagem, coluna, ordem=rotulos,
                           titulo=f'Taxa de inadimplência por quartil da quantidade de VPs com {codigo} (safras 202301 a 202308)',
                           xlabel=f'Em quantas das {len(vps_do_codigo)} VPs o cliente tem {codigo}',
                           nome_figura=f'inadimplencia_por_contagem_{nome}')

# O 9999 é código só na VP341
tem_9999 = df_des['VP341'] == 9999
inad_com = (df_des.loc[dev & tem_9999, 'adimplencia'] == 0).mean()
inad_sem = (df_des.loc[dev & ~tem_9999, 'adimplencia'] == 0).mean()
print(f'Inadimplência com 9999 na VP341: {inad_com:.1%}; sem 9999: {inad_sem:.1%}')

**Conclusão**

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

Em qualquer caso, os códigos não são convertidos em nulo nem misturados com os valores normais. No notebook `02_selecao_variaveis`, cada código fica numa faixa própria no WoE e no PSI, e a ficha das variáveis mostra `pct_-1`, `pct_-2` e `pct_9999`, a fração das linhas de cada VP com cada código.

### 2.6 Perfil dos clientes

[IA] - Utilizamos IA para transformar o código das tabelas e gráficos das distribuições das variáveis na função reutilizável `distribuicao_frequencia`, a fim de tornar a visualização e o notebook mais limpo. A função fica em `src/equifax/viz.py`.

#### 2.6.1 Adimplência geral

In [ ]:
# Frequência de adimplência: 0 = inadimplente, 1 = adimplente
distribuicao_frequencia(df_des, 'adimplencia',
                        titulo='Distribuição de adimplência', rotulo='adimplencia', nome_figura='dist_adimplencia')

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

#### 2.6.2 Distribuição de clientes por safra

In [ ]:
# Frequência de safra (número de concessões em cada mês)
distribuicao_frequencia(df_des, 'safra',
                        titulo='Distribuição de safra', rotulo='Safra', nome_figura='dist_safra')

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

#### 2.6.3 Evolução temporal da adimplência (por safra)

In [ ]:
# Taxa de inadimplência por safra (adimplencia == 0 -> inadimplente)
taxa_safra = (df_des.groupby('safra')['adimplencia']
              .agg(Total='count', Inadimplentes=lambda s: (s == 0).sum()))
taxa_safra['Taxa de inadimplência'] = taxa_safra['Inadimplentes'] / taxa_safra['Total']

display(taxa_safra.style.format({'Taxa de inadimplência': '{:.2%}'}))

# Safra como texto para o eixo x ficar categórico (202301, 202302, ...)
safras = taxa_safra.index.astype(str).values
taxas = taxa_safra['Taxa de inadimplência'].values * 100
taxa_media = taxa_safra['Inadimplentes'].sum() / taxa_safra['Total'].sum()

plt.figure(figsize=(12, 7))
ax = sns.lineplot(x=safras, y=taxas, marker='o', color='steelblue')

# Valor em cima de cada ponto
for x, y in zip(safras, taxas):
    ax.annotate(f'{y:.2f}%', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

# Linha de referência com a taxa média do período
ax.axhline(taxa_media * 100, color='gray', linestyle='--', label=f'Média do período ({taxa_media:.2%})')
ax.set_ylim(0, taxas.max() * 1.2)
ax.legend()

plt.title('Evolução da taxa de inadimplência por safra')
plt.xlabel('Safra')
plt.ylabel('Taxa de inadimplência (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', color="gray", alpha=.5)
plt.tight_layout()
salvar_figura('inadimplencia_por_safra')
plt.show()

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

#### 2.6.4 Distribuição de clientes por faixa de renda

In [ ]:
# Frequência de faixa_renda

# Faixas de renda da menor para a maior (lista definida em equifax.tratamento)
ordem_rendas = ORDEM_RENDAS

distribuicao_frequencia(df_des, 'faixa_renda', ordem=ordem_rendas,
                        titulo='Distribuição de faixa_renda', rotulo='Faixa de renda', nome_figura='dist_faixa_renda')

#### 2.6.5 Distribuição de clientes por faixa de idade

In [ ]:
# Frequência de faixa_idade
distribuicao_frequencia(df_des, 'faixa_idade',
                        titulo='Distribuição de faixa_idade', rotulo='Faixa de idade', nome_figura='dist_faixa_idade')

#### 2.6.6 Distribuição de clientes por estado

In [ ]:
# Frequência de estado
ordem_estados = df_des['estado'].value_counts().index.tolist()  # do estado com mais clientes para o com menos

distribuicao_frequencia(df_des, 'estado', ordem=ordem_estados, horizontal=True,
                        titulo='Distribuição de estado', rotulo='Estado', nome_figura='dist_estado')

#### 2.6.7 Distribuição de faixa de renda por faixa de idade

In [ ]:
# Tabela
tabela = pd.crosstab(df_des['faixa_idade'], df_des['faixa_renda'], normalize='index')
tabela = tabela.reindex(columns=ordem_rendas)

display(tabela.style.format('{:.1%}'))

# Mapa de calor, com o número de clientes de cada linha no rótulo
n_por_linha = df_des['faixa_idade'].value_counts()
rotulos = []

for categoria in tabela.index:
    rotulos.append(f'{categoria} (n={n_por_linha[categoria]})')

plt.figure(figsize=(12, 7))
sns.heatmap(tabela * 100, annot=True, fmt='.1f', cmap='Blues', yticklabels=rotulos,
            cbar_kws={'label': '% dos clientes da linha'})

plt.xlabel('Faixa de renda')
plt.ylabel('Faixa de idade')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title('Distribuição de faixa de renda por faixa de idade')
plt.tight_layout()
salvar_figura('dist_renda_por_idade')
plt.show()

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

#### 2.6.8 Perfil da população por safra

A inadimplência por safra é estável (seção 2.6.3). Aqui vemos se o tipo de cliente também é: a participação de cada faixa de renda, faixa de idade e estado em cada safra. Se a composição mudasse muito de um mês para outro, uma mudança na inadimplência poderia vir do perfil, e não do risco.

In [ ]:
# % dos clientes de cada safra em cada categoria
ordens = {'faixa_renda': ordem_rendas,
          'faixa_idade': sorted(df_des['faixa_idade'].unique()),
          'estado': ordem_estados}
nomes = {'faixa_renda': 'Faixa de renda', 'faixa_idade': 'Faixa de idade', 'estado': 'Estado'}

n_por_safra = df_des['safra'].value_counts().sort_index()
rotulos = [f'{s} (n={n})' for s, n in n_por_safra.items()]
variacao = {}

for variavel, ordem in ordens.items():
    tabela = pd.crosstab(df_des['safra'], df_des[variavel], normalize='index').reindex(columns=ordem)
    variacao[nomes[variavel]] = (tabela.max() - tabela.min()) * 100

    plt.figure(figsize=(18 if variavel == 'estado' else 12, 6))
    sns.heatmap(tabela * 100, annot=True, fmt='.1f', cmap='Blues', yticklabels=rotulos,
                cbar_kws={'label': '% dos clientes da safra'})

    plt.xlabel(nomes[variavel])
    plt.ylabel('Safra')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.title(f'{nomes[variavel]} por safra')
    plt.tight_layout()
    salvar_figura(f'dist_{variavel}_por_safra')
    plt.show()

# Maior diferença entre as safras na participação de cada categoria, em pontos percentuais
for nome, v in variacao.items():
    print(f'{nome}: maior diferença entre safras de {v.max():.1f} p.p. ({v.idxmax()})')

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

### 2.7 Análise de adimplência

[IA] - Novamente, utilizamos IA para transformar o código das tabelas e gráficos do cruzamento das variáveis com a adimplência na função reutilizável `cruzar_com_adimplencia`, a fim de tornar a visualização e o notebook mais limpo. A função fica em `src/equifax/viz.py`.

#### 2.7.1 Adimplência por faixa de renda

In [ ]:
# Taxa de inadimplência em cada faixa_renda
cruzar_com_adimplencia(df_des, 'faixa_renda', ordem=ordem_rendas,
                       titulo='Taxa de inadimplência por faixa de renda',
                       xlabel='Faixa de renda', nome_figura='inadimplencia_por_faixa_renda')

#### 2.7.2 Adimplência por faixa_idade

In [ ]:
# Taxa de inadimplência em cada faixa_idade
cruzar_com_adimplencia(df_des, 'faixa_idade',
                       titulo='Taxa de inadimplência por faixa de idade',
                       xlabel='Faixa de idade', nome_figura='inadimplencia_por_faixa_idade')

#### 2.7.3 Adimplência por estado

In [ ]:
# Taxa de inadimplência em cada estado
cruzar_com_adimplencia(df_des, 'estado', horizontal=True,
                       titulo='Taxa de inadimplência por estado',
                       xlabel='Estado', nome_figura='inadimplencia_por_estado',ordem='decrescente')

### 2.8 Inadimplência por perfil

[IA] - `cruzar_com_renda_e_adimplencia` (em `src/equifax/viz.py`) também é mais uma função reutilizável feita com auxílio de IA. Ela cruza uma variável com a faixa de renda e mostra a taxa de inadimplência de cada combinação, em tabela e mapa de calor. Cada célula mostra a taxa e, embaixo, o número de clientes (n). As células com menos de 30 clientes ficam em cinza: nelas, um ou dois inadimplentes a mais mudam muito a taxa, então o valor não deve ser lido como o risco daquela combinação.

#### 2.8.1 Faixa de renda × faixa de idade x adimplência

In [ ]:
# Taxa de inadimplência (%) em cada combinação de faixa de idade e faixa de renda
cruzar_com_renda_e_adimplencia(df_des, 'faixa_idade', ordem_rendas=ordem_rendas,
                               titulo='Taxa de inadimplência por faixa de idade e faixa de renda',
                               rotulo='Faixa de idade', nome_figura='inadimplencia_por_idade_e_renda')

**Células pequenas** \
Nas combinações com poucos clientes a taxa fica instável: um ou dois inadimplentes a mais mudam muito o valor. Por isso as células com menos de 30 clientes ficam em cinza e não entram na leitura.

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

### 2.9 Estabilidade no tempo

A estabilidade das variáveis (PSI) e o poder preditivo (WoE e IV) são analisados no notebook `02_selecao_variaveis.ipynb`, junto da seleção de variáveis (Passo 4 do roteiro), porque as duas análises usam a mesma divisão de treino e teste. Neste notebook fica apenas a estabilidade da taxa de inadimplência por safra (seção 2.6.3).

### 2.10 Salvar CSV

A base com os tratamentos das seções 2.2 a 2.4 é salva em `data/processed/base_des_01.csv` (pasta fora do Git). Ela é a entrada do `02_selecao_variaveis.ipynb`.

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df_des.to_csv(DATA_PROCESSED / 'base_des_01.csv', index=False)